In [1]:
import pandas as pd
import numpy as np
import os

data_path = os.path.expanduser('~/Desktop/football-intelligence/data/')

# Chargement des données nettoyées
player_profile = pd.read_csv(data_path + 'player_profile_clean.csv')
clubs_top5 = pd.read_csv(data_path + 'clubs_top5_clean.csv')

print(f"✅ Joueurs chargés : {len(player_profile):,}")
print(f"✅ Clubs chargés : {len(clubs_top5)}")
print(f"\nColonnes player_profile : {list(player_profile.columns)}")

✅ Joueurs chargés : 10,101
✅ Clubs chargés : 176

Colonnes player_profile : ['player_id', 'total_appearances', 'total_goals', 'total_assists', 'total_minutes', 'total_yellow', 'total_red', 'goals_per90', 'assists_per90', 'minutes_per_game', 'name_player', 'position', 'sub_position', 'market_value_in_eur', 'highest_market_value_in_eur', 'current_club_id', 'date_of_birth', 'foot', 'height_in_cm', 'country_of_citizenship', 'current_club_domestic_competition_id', 'age', 'club_id', 'name_club', 'domestic_competition_id', 'squad_size', 'average_age', 'total_market_value', 'net_transfer_record']


In [2]:
# ============================================
# FEATURE ENGINEERING - PROFIL JOUEUR
# ============================================

df = player_profile.copy()

# 1. Score offensif (attaquants/milieux)
df['offensive_score'] = (
    df['goals_per90'] * 0.6 + 
    df['assists_per90'] * 0.4
).round(3)

# 2. Score de régularité (temps de jeu)
df['regularity_score'] = (
    df['minutes_per_game'] / 90
).clip(0, 1).round(3)

# 3. Score de discipline (cartons)
df['discipline_score'] = (
    1 - (df['total_yellow'] * 0.02 + df['total_red'] * 0.1) / 
    df['total_appearances'].clip(1)
).clip(0, 1).round(3)

# 4. Potentiel non exploité (valeur actuelle vs valeur max)
df['potential_ratio'] = (
    df['market_value_in_eur'] / 
    df['highest_market_value_in_eur'].clip(1)
).clip(0, 1).round(3)

# 5. Encodage de la position
position_map = {
    'Attack': 4, 
    'Midfield': 3, 
    'Defender': 2, 
    'Goalkeeper': 1
}
df['position_code'] = df['position'].map(position_map).fillna(0)

# 6. Encodage du pied
df['foot_code'] = df['foot'].map({'right': 1, 'left': 2, 'both': 3}).fillna(0)

print("✅ Features joueurs créées !")
print(df[['name_player', 'position', 'offensive_score', 
          'regularity_score', 'discipline_score', 
          'potential_ratio']].head(10).to_string())

✅ Features joueurs créées !
           name_player    position  offensive_score  regularity_score  discipline_score  potential_ratio
0       Miroslav Klose      Attack            0.418             0.734             0.997            0.033
1   Roman Weidenfeller  Goalkeeper            0.000             0.989             0.999            0.094
2     Dimitar Berbatov      Attack            0.316             0.864             0.998            0.029
3                Lúcio    Defender            0.000             1.000             1.000            0.008
4           Tom Starke  Goalkeeper            0.000             1.000             1.000            0.033
5  Christoph Metzelder    Defender            0.000             0.561             1.000            0.158
6        Tomas Rosicky    Midfield            0.160             0.590             0.996            0.020
7     Roque Santa Cruz      Attack            0.252             0.620             0.999            0.021
8       Gerald Asamoah     

In [3]:
# ============================================
# FEATURE ENGINEERING - PROFIL CLUB
# ============================================

clubs_df = clubs_top5.copy()

# 1. Valeur totale de l'effectif (normalisation)
clubs_df['total_market_value'] = pd.to_numeric(
    clubs_df['total_market_value'], errors='coerce'
).fillna(0)

clubs_df['squad_value_normalized'] = (
    clubs_df['total_market_value'] / clubs_df['total_market_value'].max()
).round(3)

# 2. Taille de l'effectif (équipes compétitives = 20-30 joueurs)
clubs_df['squad_size'] = pd.to_numeric(clubs_df['squad_size'], errors='coerce').fillna(0)
clubs_df['squad_size_score'] = (
    1 - abs(clubs_df['squad_size'] - 25) / 25
).clip(0, 1).round(3)

# 3. Âge moyen de l'équipe
clubs_df['average_age'] = pd.to_numeric(clubs_df['average_age'], errors='coerce').fillna(25)

# 4. Encodage de la ligue
league_prestige = {'GB1': 5, 'ES1': 5, 'L1': 4, 'IT1': 4, 'FR1': 3}
clubs_df['league_prestige'] = clubs_df['domestic_competition_id'].map(league_prestige).fillna(1)

# 5. Nombre de joueurs étrangers (ouverture internationale)
clubs_df['foreigners_percentage'] = pd.to_numeric(
    clubs_df['foreigners_percentage'], errors='coerce'
).fillna(0)

print("✅ Features clubs créées !")
print(clubs_df[['name', 'domestic_competition_id', 'squad_value_normalized',
                'squad_size_score', 'average_age', 
                'league_prestige', 'foreigners_percentage']].head(10).to_string())

✅ Features clubs créées !
                         name domestic_competition_id  squad_value_normalized  squad_size_score  average_age  league_prestige  foreigners_percentage
0           Arminia Bielefeld                      L1                     NaN              0.92         25.3                4                   55.6
1         Paris Football Club                     FR1                     NaN              0.76         28.6                3                   54.8
2              Leicester City                     GB1                     NaN              0.84         25.9                5                   58.6
3       Unione Sportiva Lecce                     IT1                     NaN              0.92         25.2                4                   85.2
4                  Watford FC                     GB1                     NaN              0.80         26.3                5                   80.0
5  Bologna Football Club 1909                     IT1                     NaN   

In [4]:
# Correction de total_market_value (format texte → nombre)
print("Exemple valeurs brutes:", clubs_top5['total_market_value'].head(5).tolist())

Exemple valeurs brutes: [nan, nan, nan, nan, nan]


In [5]:
# Calcul de la valeur totale du club à partir des joueurs
club_market_value = df.groupby('current_club_id')['market_value_in_eur'].sum().reset_index()
club_market_value.columns = ['club_id', 'calculated_market_value']

# Fusion avec clubs_df
clubs_df = clubs_df.merge(club_market_value, on='club_id', how='left')
clubs_df['calculated_market_value'] = clubs_df['calculated_market_value'].fillna(0)

# Normalisation de la valeur calculée
clubs_df['squad_value_normalized'] = (
    clubs_df['calculated_market_value'] / clubs_df['calculated_market_value'].max()
).round(3)

print("✅ Valeurs marchandes des clubs recalculées !")
print(clubs_df[['name', 'domestic_competition_id', 'calculated_market_value', 
                'squad_value_normalized', 'league_prestige']].sort_values(
    'calculated_market_value', ascending=False).head(10).to_string())

✅ Valeurs marchandes des clubs recalculées !
                                  name domestic_competition_id  calculated_market_value  squad_value_normalized  league_prestige
121         Real Madrid Club de Fútbol                     ES1             1.303500e+09                   1.000                5
78       Manchester City Football Club                     GB1             1.303200e+09                   1.000                5
135  Paris Saint-Germain Football Club                     FR1             1.254150e+09                   0.962                3
143              Chelsea Football Club                     GB1             1.204150e+09                   0.924                5
38               Futbol Club Barcelona                     ES1             1.193025e+09                   0.915                5
19               Arsenal Football Club                     GB1             1.150950e+09                   0.883                5
87             Liverpool Football Club              

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

# ============================================
# ALGORITHME DE RECOMMANDATION DE CLUBS
# ============================================

# Features du joueur pour le matching
player_features = ['offensive_score', 'regularity_score', 'discipline_score', 
                   'potential_ratio', 'position_code', 'foot_code',
                   'market_value_in_eur', 'age']

# Features du club pour le matching  
club_features = ['squad_value_normalized', 'squad_size_score', 
                 'average_age', 'league_prestige', 'foreigners_percentage']

# Nettoyage
df_model = df[player_features + ['player_id', 'name_player', 'position', 
                                  'current_club_id', 'name_club']].dropna()
clubs_model = clubs_df[club_features + ['club_id', 'name', 
                                         'domestic_competition_id']].dropna()

print(f"✅ Joueurs prêts pour le modèle : {len(df_model):,}")
print(f"✅ Clubs prêts pour le modèle : {len(clubs_model)}")

# Normalisation des features
scaler_player = StandardScaler()
scaler_club = StandardScaler()

player_scaled = scaler_player.fit_transform(df_model[player_features])
club_scaled = scaler_club.fit_transform(clubs_model[club_features])

print("\n✅ Normalisation effectuée !")
print(f"Shape joueurs : {player_scaled.shape}")
print(f"Shape clubs : {club_scaled.shape}")

✅ Joueurs prêts pour le modèle : 7,148
✅ Clubs prêts pour le modèle : 176

✅ Normalisation effectuée !
Shape joueurs : (7148, 8)
Shape clubs : (176, 5)


In [7]:
# ============================================
# FONCTION DE RECOMMANDATION - TOP 5 CLUBS
# ============================================

def recommend_clubs(player_name, top_n=5):
    """
    Recommande les top N clubs pour un joueur donné.
    """
    # Recherche du joueur
    player_row = df_model[df_model['name_player'].str.lower() == player_name.lower()]
    
    if player_row.empty:
        # Recherche partielle si nom exact non trouvé
        player_row = df_model[df_model['name_player'].str.lower().str.contains(player_name.lower())]
        if player_row.empty:
            print(f"❌ Joueur '{player_name}' non trouvé.")
            return None
        print(f"🔍 Joueur trouvé : {player_row.iloc[0]['name_player']}")
    
    player_row = player_row.iloc[0]
    
    # Récupération des features du joueur
    player_vals = player_row[player_features].values.reshape(1, -1)
    player_vals_scaled = scaler_player.transform(player_vals)
    
    # Calcul de compatibilité avec chaque club
    # On utilise les 5 features joueur les plus compatibles avec les clubs
    player_for_clubs = player_vals_scaled[:, :5]  # offensive, regularity, discipline, potential, position
    
    similarities = cosine_similarity(player_for_clubs, club_scaled)[0]
    
    # Top N clubs
    top_indices = similarities.argsort()[::-1][:top_n]
    
    print(f"\n🏆 TOP {top_n} CLUBS RECOMMANDÉS POUR : {player_row['name_player']}")
    print(f"   Position : {player_row['position']} | Âge : {int(player_row['age'])} ans")
    print(f"   Valeur marchande : {player_row['market_value_in_eur']:,.0f} €")
    print(f"   Club actuel : {player_row['name_club']}")
    print("-" * 60)
    
    results = []
    for rank, idx in enumerate(top_indices, 1):
        club = clubs_model.iloc[idx]
        score = round(similarities[idx] * 100, 1)
        print(f"  {rank}. {club['name']} ({club['domestic_competition_id']}) — Compatibilité : {score}%")
        results.append({
            'rank': rank,
            'club': club['name'],
            'league': club['domestic_competition_id'],
            'compatibility': score
        })
    
    return results

# TEST avec un vrai joueur
recommend_clubs("Kylian Mbappé")


🏆 TOP 5 CLUBS RECOMMANDÉS POUR : Kylian Mbappé
   Position : Attack | Âge : 27 ans
   Valeur marchande : 200,000,000 €
   Club actuel : Real Madrid Club de Fútbol
------------------------------------------------------------
  1. Manchester United Football Club (GB1) — Compatibilité : 97.8%
  2. Association Football Club Bournemouth (GB1) — Compatibilité : 95.1%
  3. Brentford Football Club (GB1) — Compatibilité : 95.0%
  4. Tottenham Hotspur Football Club (GB1) — Compatibilité : 94.7%
  5. Crystal Palace Football Club (GB1) — Compatibilité : 94.4%


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


[{'rank': 1,
  'club': 'Manchester United Football Club',
  'league': 'GB1',
  'compatibility': np.float64(97.8)},
 {'rank': 2,
  'club': 'Association Football Club Bournemouth',
  'league': 'GB1',
  'compatibility': np.float64(95.1)},
 {'rank': 3,
  'club': 'Brentford Football Club',
  'league': 'GB1',
  'compatibility': np.float64(95.0)},
 {'rank': 4,
  'club': 'Tottenham Hotspur Football Club',
  'league': 'GB1',
  'compatibility': np.float64(94.7)},
 {'rank': 5,
  'club': 'Crystal Palace Football Club',
  'league': 'GB1',
  'compatibility': np.float64(94.4)}]

In [8]:
# VERSION AMÉLIORÉE - Matching plus intelligent
def recommend_clubs_v2(player_name, top_n=5):
    """
    Recommandation améliorée avec pondération valeur marchande + diversité des ligues.
    """
    player_row = df_model[df_model['name_player'].str.lower().str.contains(player_name.lower())]
    
    if player_row.empty:
        print(f"❌ Joueur '{player_name}' non trouvé.")
        return None
    
    player_row = player_row.iloc[0]
    player_vals = pd.DataFrame([player_row[player_features]])
    player_vals_scaled = scaler_player.transform(player_vals)
    player_for_clubs = player_vals_scaled[:, :5]
    
    similarities = cosine_similarity(player_for_clubs, club_scaled)[0]
    
    # Pondération par compatibilité de valeur marchande
    player_value = player_row['market_value_in_eur']
    clubs_model_copy = clubs_model.copy()
    clubs_model_copy['similarity'] = similarities
    clubs_model_copy['value_ratio'] = clubs_df['calculated_market_value'].values / (player_value * 25 + 1)
    clubs_model_copy['value_score'] = (1 - abs(1 - clubs_model_copy['value_ratio'])).clip(0, 1)
    
    # Score final combiné
    clubs_model_copy['final_score'] = (
        clubs_model_copy['similarity'] * 0.6 + 
        clubs_model_copy['value_score'] * 0.4
    )
    
    # Exclure le club actuel
    current_club = player_row['current_club_id']
    clubs_filtered = clubs_model_copy[clubs_model_copy['club_id'] != current_club]
    
    # Top N en diversifiant les ligues (max 2 clubs par ligue)
    results = []
    league_count = {}
    for _, row in clubs_filtered.sort_values('final_score', ascending=False).iterrows():
        league = row['domestic_competition_id']
        league_count[league] = league_count.get(league, 0) + 1
        if league_count[league] <= 2:
            results.append(row)
        if len(results) == top_n:
            break
    
    print(f"\n🏆 TOP {top_n} CLUBS RECOMMANDÉS POUR : {player_row['name_player']}")
    print(f"   Position : {player_row['position']} | Âge : {int(player_row['age'])} ans")
    print(f"   Valeur marchande : {player_value:,.0f} €")
    print(f"   Club actuel : {player_row['name_club']}")
    print("-" * 60)
    
    final_results = []
    for rank, row in enumerate(results, 1):
        score = round(row['final_score'] * 100, 1)
        print(f"  {rank}. {row['name']} ({row['domestic_competition_id']}) — Score : {score}%")
        final_results.append({'rank': rank, 'club': row['name'], 
                               'league': row['domestic_competition_id'], 'score': score})
    
    return final_results

# TEST
recommend_clubs_v2("Kylian Mbappé")


🏆 TOP 5 CLUBS RECOMMANDÉS POUR : Kylian Mbappé
   Position : Attack | Âge : 27 ans
   Valeur marchande : 200,000,000 €
   Club actuel : Real Madrid Club de Fútbol
------------------------------------------------------------
  1. Manchester United Football Club (GB1) — Score : 64.9%
  2. Arsenal Football Club (GB1) — Score : 64.7%
  3. Futbol Club Barcelona (ES1) — Score : 58.0%
  4. Club Atlético de Madrid S.A.D. (ES1) — Score : 56.0%
  5. FC Bayern München (L1) — Score : 47.7%


[{'rank': 1,
  'club': 'Manchester United Football Club',
  'league': 'GB1',
  'score': 64.9},
 {'rank': 2, 'club': 'Arsenal Football Club', 'league': 'GB1', 'score': 64.7},
 {'rank': 3, 'club': 'Futbol Club Barcelona', 'league': 'ES1', 'score': 58.0},
 {'rank': 4,
  'club': 'Club Atlético de Madrid S.A.D.',
  'league': 'ES1',
  'score': 56.0},
 {'rank': 5, 'club': 'FC Bayern München', 'league': 'L1', 'score': 47.7}]

In [9]:
# TESTS DE VALIDATION
print("=" * 60)
recommend_clubs_v2("Erling Haaland")

print("\n" + "=" * 60)
recommend_clubs_v2("Lamine Yamal")

print("\n" + "=" * 60)
recommend_clubs_v2("Thibaut Courtois")


🏆 TOP 5 CLUBS RECOMMANDÉS POUR : Erling Haaland
   Position : Attack | Âge : 25 ans
   Valeur marchande : 200,000,000 €
   Club actuel : Manchester City Football Club
------------------------------------------------------------
  1. Manchester United Football Club (GB1) — Score : 65.1%
  2. Arsenal Football Club (GB1) — Score : 64.1%
  3. Real Madrid Club de Fútbol (ES1) — Score : 58.4%
  4. Club Atlético de Madrid S.A.D. (ES1) — Score : 58.1%
  5. Juventus Football Club (IT1) — Score : 49.4%


🏆 TOP 5 CLUBS RECOMMANDÉS POUR : Lamine Yamal
   Position : Attack | Âge : 18 ans
   Valeur marchande : 200,000,000 €
   Club actuel : Futbol Club Barcelona
------------------------------------------------------------
  1. West Ham United Football Club (GB1) — Score : 62.3%
  2. Brighton and Hove Albion Football Club (GB1) — Score : 61.0%
  3. Club Atlético de Madrid S.A.D. (ES1) — Score : 59.2%
  4. Real Madrid Club de Fútbol (ES1) — Score : 47.5%
  5. Associazione Calcio Milan (IT1) — Score :

[{'rank': 1,
  'club': 'Associazione Calcio Fiorentina',
  'league': 'IT1',
  'score': 67.4},
 {'rank': 2, 'club': 'Sport-Club Freiburg', 'league': 'L1', 'score': 62.6},
 {'rank': 3, 'club': 'Athletic Club Bilbao', 'league': 'ES1', 'score': 61.3},
 {'rank': 4, 'club': 'FC Crotone', 'league': 'IT1', 'score': 58.5},
 {'rank': 5,
  'club': 'Real Sociedad de Fútbol S.A.D.',
  'league': 'ES1',
  'score': 56.7}]

In [10]:
# TESTS avec joueurs de clubs moyens
print("=" * 60)
recommend_clubs_v2("Alexandre Lacazette")

print("\n" + "=" * 60)
recommend_clubs_v2("Florian Thauvin")

print("\n" + "=" * 60)
recommend_clubs_v2("Kevin Gameiro")

❌ Joueur 'Alexandre Lacazette' non trouvé.


🏆 TOP 5 CLUBS RECOMMANDÉS POUR : Florian Thauvin
   Position : Attack | Âge : 33 ans
   Valeur marchande : 5,000,000 €
   Club actuel : Racing Club de Lens
------------------------------------------------------------
  1. Football Club Lorient-Bretagne Sud (FR1) — Score : 82.2%
  2. Udinese Calcio (IT1) — Score : 79.5%
  3. Genoa Cricket and Football Club (IT1) — Score : 79.0%
  4. Toulouse Football Club (FR1) — Score : 74.2%
  5. 1. Fußballclub Union Berlin (L1) — Score : 70.8%

❌ Joueur 'Kevin Gameiro' non trouvé.


In [11]:
# Sauvegarde du dataset final avec toutes les features
df_model_save = df.copy()
df_model_save.to_csv(data_path + 'player_features_final.csv', index=False)
clubs_df.to_csv(data_path + 'clubs_features_final.csv', index=False)

print("✅ Datasets finaux sauvegardés !")
print(f"   - player_features_final.csv ({len(df_model_save):,} joueurs)")
print(f"   - clubs_features_final.csv ({len(clubs_df)} clubs)")

# Sauvegarde du modèle avec pickle
import pickle

model_data = {
    'df_model': df_model,
    'clubs_model': clubs_model,
    'clubs_df': clubs_df,
    'scaler_player': scaler_player,
    'scaler_club': scaler_club,
    'player_features': player_features,
    'club_features': club_features
}

with open(data_path + 'recommendation_model.pkl', 'wb') as f:
    pickle.dump(model_data, f)

print("   - recommendation_model.pkl ✅")
print("\n🎯 Feature Engineering terminé — prêt pour Streamlit !")

✅ Datasets finaux sauvegardés !
   - player_features_final.csv (10,101 joueurs)
   - clubs_features_final.csv (176 clubs)
   - recommendation_model.pkl ✅

🎯 Feature Engineering terminé — prêt pour Streamlit !


In [12]:
import pickle
import os

data_path = os.path.expanduser('~/Desktop/football-intelligence/data/')

# On resauvegarde avec TOUTES les colonnes de df
model_data_updated = {
    'df_model': df,  # df complet avec toutes les stats
    'clubs_model': clubs_model,
    'clubs_df': clubs_df,
    'scaler_player': scaler_player,
    'scaler_club': scaler_club,
    'player_features': player_features,
    'club_features': club_features
}

with open(data_path + 'recommendation_model.pkl', 'wb') as f:
    pickle.dump(model_data_updated, f)

print("✅ Modèle mis à jour avec toutes les colonnes !")
print(f"Colonnes disponibles : {list(df.columns)}")

✅ Modèle mis à jour avec toutes les colonnes !
Colonnes disponibles : ['player_id', 'total_appearances', 'total_goals', 'total_assists', 'total_minutes', 'total_yellow', 'total_red', 'goals_per90', 'assists_per90', 'minutes_per_game', 'name_player', 'position', 'sub_position', 'market_value_in_eur', 'highest_market_value_in_eur', 'current_club_id', 'date_of_birth', 'foot', 'height_in_cm', 'country_of_citizenship', 'current_club_domestic_competition_id', 'age', 'club_id', 'name_club', 'domestic_competition_id', 'squad_size', 'average_age', 'total_market_value', 'net_transfer_record', 'offensive_score', 'regularity_score', 'discipline_score', 'potential_ratio', 'position_code', 'foot_code']
